In [1]:
import numpy as np
import math
import random

from sklearn.neural_network import MLPRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error

# ============================================================
# WEEK 9 — FUNCTION 2 (ROBUST LOCAL-MAXIMISATION v3)
# Fixes + upgrades:
#  - Fix X syntax errors + ensure 18 points align with y
#  - Deduplication kept (averages y for identical X)
#  - More robust acquisition: mix of UCB + EI + Thompson
#  - Trust-region + global exploration candidates
#  - 6-decimal fixed output formatting
# ============================================================

# ============================================================
# 1. INPUT DATA (18 points)
#    NOTE: Your original X block had:
#      - a missing closing bracket/comma around [0.99947700, 0.02152800]
#      - a duplicated [0.99145700, 0.00174400]
#    We keep duplicates intentionally (dedupe will average y).
# ============================================================

X = np.array([
    [0.66579958, 0.12396913],
    [0.87779099, 0.77862750],
    [0.14269907, 0.34900513],
    [0.84527543, 0.71112027],
    [0.45464714, 0.29045518],
    [0.57771284, 0.77197318],
    [0.43816606, 0.68501826],
    [0.34174959, 0.02869772],
    [0.33864816, 0.21386725],
    [0.70263656, 0.92656420],
    [0.92656400, 1.02656400],   # will be clipped to 1.0
    [0.74750400, 0.20897000],
    [0.68340600, 0.06376900],
    [0.58313700, 0.01254900],
    [0.99145700, 0.00174400],
    [0.99145700, 0.00174400],   # duplicate
    [0.99947700, 0.02152800],
    [0.00757800, 0.97735900]
], dtype=float)

y = np.array([
    0.53899612, 0.42058624, -0.06562362, 0.29399291, 0.21496451,
    0.02310555, 0.24461934, 0.03874902, -0.01385762, 0.61120522,
    -0.04199554, 0.28033031, 0.62973064, 0.06695726,
    0.11670354823827367, 0.11353912028668156, 0.02426575623315439,
    0.15170334833240093
], dtype=float)

# Clip to [0,1]^2 as per black-box bounds
X = np.clip(X, 0.0, 1.0)

# ============================================================
# 2. DEDUPLICATION (average y for identical X)
# ============================================================

def dedupe_average(X, y):
    b = np.ascontiguousarray(X).view(
        np.dtype((np.void, X.dtype.itemsize * X.shape[1]))
    )
    _, inv = np.unique(b, return_inverse=True)

    Xu, yu = [], []
    for i in np.unique(inv):
        idx = np.where(inv == i)[0]
        Xu.append(X[idx[0]])
        yu.append(y[idx].mean())
    return np.array(Xu), np.array(yu)

X, y = dedupe_average(X, y)

# ============================================================
# 3. TRAIN-ONLY NOISE (regularises small-data MLPs)
# ============================================================

class TrainOnlyNoise:
    def __init__(self, sigma=0.005, seed=0):
        self.sigma = float(sigma)
        self.seed = int(seed)

    def fit(self, X, y=None):
        return self

    def fit_transform(self, X, y=None):
        rng = np.random.default_rng(self.seed)
        return X + rng.normal(0.0, self.sigma, size=X.shape)

    def transform(self, X):
        return X

def make_model(seed,
               hidden=(64, 32),
               alpha=5e-5,
               lr=0.01,
               sigma=0.005,
               max_iter=5000,
               tol=1e-7,
               n_iter_no_change=40):
    return Pipeline([
        ("scaler", StandardScaler()),
        ("noise", TrainOnlyNoise(sigma=sigma, seed=seed)),
        ("mlp", MLPRegressor(
            hidden_layer_sizes=hidden,
            activation="relu",
            solver="adam",
            learning_rate="adaptive",
            learning_rate_init=lr,
            alpha=alpha,
            early_stopping=True,
            n_iter_no_change=n_iter_no_change,
            max_iter=max_iter,
            tol=tol,
            random_state=seed
        ))
    ])

# ============================================================
# 4. CV RANDOM SEARCH (slightly more cost-aware for Week 9)
#    With 18 pts, we reduce trials a bit and lean on robust acquisition.
# ============================================================

def cv_mse_for_config(X, y, cfg):
    kf = KFold(n_splits=5, shuffle=True, random_state=123)
    mses = []
    for tr, te in kf.split(X):
        preds = []
        for i in range(cfg["ens_cv"]):
            m = make_model(
                seed=1000 + i,
                hidden=cfg["hidden"],
                alpha=cfg["alpha"],
                lr=cfg["lr"],
                sigma=cfg["sigma"],
                max_iter=cfg["max_iter"],
                tol=cfg["tol"],
                n_iter_no_change=cfg["ninc"]
            )
            m.fit(X[tr], y[tr])
            preds.append(m.predict(X[te]))
        mses.append(mean_squared_error(y[te], np.mean(preds, axis=0)))
    return float(np.mean(mses))

def random_search_best_config(X, y, n_trials=40, seed=9):
    # Week 9: slightly fewer trials than Week 8, to trade cost for robustness elsewhere
    random.seed(seed)
    best_cfg, best_mse = None, float("inf")

    for _ in range(n_trials):
        cfg = {
            "hidden": random.choice([(32, 16), (64, 32), (64, 64), (128, 64)]),
            "alpha": random.choice([1e-6, 1e-5, 5e-5, 1e-4]),
            "lr": random.choice([3e-3, 1e-2, 2e-2]),
            "sigma": random.choice([0.0, 0.003, 0.005, 0.01]),
            "tol": random.choice([1e-6, 1e-7]),
            "ninc": random.choice([30, 40, 60]),
            "max_iter": random.choice([3000, 5000]),
            "ens_cv": 5
        }
        mse = cv_mse_for_config(X, y, cfg)
        if mse < best_mse:
            best_cfg, best_mse = cfg, mse
    return best_cfg, best_mse

best_cfg, best_cv_mse = random_search_best_config(X, y)

# ============================================================
# 5. FINAL ENSEMBLE (bagged seeds for robustness)
# ============================================================

def fit_ensemble(X, y, cfg, n=30):
    models = []
    for i in range(n):
        m = make_model(
            seed=300 + i,
            hidden=cfg["hidden"],
            alpha=cfg["alpha"],
            lr=cfg["lr"],
            sigma=cfg["sigma"],
            max_iter=cfg["max_iter"],
            tol=cfg["tol"],
            n_iter_no_change=cfg["ninc"]
        )
        m.fit(X, y)
        models.append(m)
    return models

def ensemble_predict(models, Xq):
    preds = np.vstack([m.predict(Xq) for m in models])  # (n_models, n_pts)
    mu = preds.mean(axis=0)
    std = preds.std(axis=0, ddof=1) + 1e-9
    return mu, std, preds

models = fit_ensemble(X, y, best_cfg, n=30)

# ============================================================
# 6. ACQUISITION (Week 9: robust mix + trust region)
#    - EI encourages improvement over y_best
#    - UCB balances mean/uncertainty
#    - Thompson adds randomness to avoid brittle local traps
# ============================================================

def erf_vec(x):
    return np.vectorize(math.erf)(x)

def compute_ei(mu, std, y_best, xi=0.001):
    z = (mu - y_best - xi) / std
    pdf = np.exp(-0.5 * z * z) / np.sqrt(2 * np.pi)
    cdf = 0.5 * (1 + erf_vec(z / np.sqrt(2)))
    return (mu - y_best - xi) * cdf + std * pdf

idx_best = int(np.argmax(y))
x_best = X[idx_best]
y_best = float(y[idx_best])

rng = np.random.default_rng(2026)

# Scaling-law-ish heuristic: as data grows, shrink trust region.
# With ~18 pts, keep a modest TR but not too tight.
n = len(y)
tr_sigma = float(np.clip(0.08 / np.sqrt(max(n, 1)), 0.015, 0.04))  # ~0.02 for n~18

# Candidate pool: global + trust-region + "edge probing"
N_global = 25000
N_local  = 25000
N_edge   = 8000

X_global = rng.uniform(0, 1, (N_global, 2))
X_local  = np.clip(rng.normal(loc=x_best, scale=tr_sigma, size=(N_local, 2)), 0, 1)

# Edge probing sometimes finds boundary maxima in constrained boxes
edges = rng.uniform(0, 1, (N_edge, 2))
mask = rng.random(N_edge) < 0.5
edges[mask, 0] = rng.choice([0.0, 1.0], size=mask.sum())
edges[~mask, 1] = rng.choice([0.0, 1.0], size=(~mask).sum())

Xcand = np.vstack([X_global, X_local, edges])

mu, std, all_preds = ensemble_predict(models, Xcand)

# Thompson sample: sample one model's prediction per candidate (random model index)
th_idx = rng.integers(0, all_preds.shape[0], size=Xcand.shape[0])
th = all_preds[th_idx, np.arange(Xcand.shape[0])]

# EI + UCB
xi = 0.001
ei = compute_ei(mu, std, y_best, xi=xi)
kappa = 2.0
ucb = mu + kappa * std

# Normalise components (stable mixing)
def zscore(v):
    s = v.std()
    return (v - v.mean()) / (s + 1e-12)

# Week 9 mix:
# - slightly more weight on robustness (UCB) + some EI
# - add Thompson to reduce “brittle optimism”
score = 0.50 * zscore(ucb) + 0.30 * zscore(ei) + 0.20 * zscore(th)

# Distance filter (avoid re-querying near existing points)
dists = np.sqrt(((Xcand[:, None, :] - X[None, :, :]) ** 2).sum(axis=2))
min_dist = dists.min(axis=1)

# As data increases, you can afford a slightly smaller min distance; keep conservative for stability
min_sep = float(np.clip(0.10 / np.sqrt(max(n, 1)), 0.010, 0.020))  # ~0.02..0.01
valid = np.where(min_dist >= min_sep)[0]

best_idx = valid[int(np.argmax(score[valid]))]

# ============================================================
# 7. FINAL 6-DECIMAL OUTPUT (fixed formatting)
# ============================================================

x_next = np.round(Xcand[best_idx], 6)

print("================================================")
print("WEEK 9 FUNCTION 2 — NEXT DATA POINT (6 DECIMALS)")
print("================================================")
print(f"best_cfg (cv_mse={best_cv_mse:.6f}) = {best_cfg}")
print(f"x_best = [{x_best[0]:.6f}, {x_best[1]:.6f}], y_best = {y_best:.6f}")
print(f"x_next = [{x_next[0]:.6f}, {x_next[1]:.6f}]")
print(f"mu(x_next)  = {mu[best_idx]:.6f}")
print(f"sig(x_next) = {std[best_idx]:.6f}")
print(f"EI(x_next)  = {ei[best_idx]:.6f}")
print(f"UCB(x_next) = {ucb[best_idx]:.6f}")
print(f"min_dist    = {min_dist[best_idx]:.6f}")
print(f"trust_sigma = {tr_sigma:.6f}, min_sep = {min_sep:.6f}")


WEEK 9 FUNCTION 2 — NEXT DATA POINT (6 DECIMALS)
best_cfg (cv_mse=0.072988) = {'hidden': (64, 32), 'alpha': 1e-06, 'lr': 0.02, 'sigma': 0.003, 'tol': 1e-06, 'ninc': 60, 'max_iter': 3000, 'ens_cv': 5}
x_best = [0.683406, 0.063769], y_best = 0.629731
x_next = [1.000000, 0.982316]
mu(x_next)  = 0.262227
sig(x_next) = 0.311430
EI(x_next)  = 0.018080
UCB(x_next) = 0.885088
min_dist    = 0.075535
trust_sigma = 0.019403, min_sep = 0.020000
